In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# CrackSegDiff: Simplified Training & Inference
Automated setup, data preparation (First 500 Test / 2000 Train), training, and testing.

In [2]:
# 1. Setup Environment & Weights
!nvidia-smi
import os
if not os.path.exists('CrackSegDiff'):
    !git clone https://github.com/Ludwig-H/CrackSegDiff.git
%cd CrackSegDiff
!git pull

# Downgrade PyTorch to stable 2.4.0 for guaranteed Mamba compatibility
print("Installing PyTorch 2.4.0 compatible with Mamba wheels...")
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

!pip install -r requirement.txt

# Force install pre-built wheels for Mamba-SSM and Causal-Conv1d to avoid compilation errors and ensure GPU speed
print("Installing Optimized Mamba Kernels...")
import torch
cuda_version = torch.version.cuda.replace('.', '')
torch_version = torch.__version__.split('+')[0].replace('.', '')
# Assuming standard Colab PyTorch 2.x and CUDA 11.8 or 12.x
# We use the releases from Dao-AILab which are reliable
!pip install ninja  # Speeds up compilation
!pip install causal-conv1d>=1.0.0 --no-build-isolation -v
!pip install mamba-ssm>=1.0.1 --no-build-isolation -v

# If the above standard install fails (it tries to build), we could try finding wheels:
# (But usually --no-build-isolation helps or just standard pip works if env is clean)

# Verify installation
try:
    import mamba_ssm
    print("\nSUCCESS: Mamba SSM installed successfully! GPU acceleration enabled (x10 speed).")
except ImportError:
    print("\nWARNING: Mamba SSM installation failed.")
    print("Fallback to Pure Python active (SLOWER).")

# Download Pretrained Weights
!mkdir -p pretrained_weights
!gdown 1JYqMxM5dbCLZ-WGPKtIofYJhj0VPuy3l -O pretrained_weights/vssm_base_0229_ckpt_epoch_237.pth

# Patch Hardcoded Paths
target_file = 'CrackSegDiff/guided_diffusion/unet.py'
new_path = os.path.abspath('pretrained_weights/vssm_base_0229_ckpt_epoch_237.pth')
if os.path.exists(target_file):
    with open(target_file, 'r') as f: content = f.read()
    content = content.replace('/home/dell/jlc/segdiff/pre_trained_weights/vssm_base_0229_ckpt_epoch_237.pth', new_path)
    with open(target_file, 'w') as f: f.write(content)
    print("Path patched successfully.")

Sat Dec  6 09:09:35 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

Installing Optimized Mamba Kernels...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 5.5 MB/s eta 0:00:00
  Running command Preparing metadata (pyproject.toml)


  torch.__version__  = 2.4.0+cu121


  running dist_info
  creating /tmp/pip-modern-metadata-ciybxssh/causal_conv1d.egg-info
  writing /tmp/pip-modern-metadata-ciybxssh/causal_conv1d.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-modern-metadata-ciybxssh/causal_conv1d.egg-info/dependency_links.txt
  writing requirements to /tmp/pip-modern-metadata-ciybxssh/causal_conv1d.egg-info/requires.txt
  writing top-level names to /tmp/pip-modern-metadata-ciybxssh/causal_conv1d.egg-info/top_level.txt
  writing manifest file '/tmp/pip-modern-metadata-ciybxssh/causal_conv1d.egg-info/SOURCES.txt'
  reading manifest file '/tmp/pip-modern-metadata-ciybxssh/causal_conv1d.egg-info/SOURCES.txt'
  reading manifest template 'MANIFEST.in'

  adding license file 'LICENSE'
  adding license file 'AUTHORS'
  writing manifest file 

In [3]:
# 2. Prepare Data (First 500 Test / 2000 Train)
!rm -rf data && mkdir -p data
%cd data
!gdown 1qnLMCeon7LJjT9H0ENiNF5sFs-F7-NvK -O data.zip
!unzip -q -o data.zip
%cd ..

import os, glob, shutil
print("Organizing Data...")

# Find folders
try:
    src_img = glob.glob('data/**/img/fused', recursive=True)[0]
    src_lbl = glob.glob('data/**/lbs', recursive=True)[0]
except IndexError:
    # Fallback if 'fused' not found, try 'intensity'
    print("Fused folder not found, checking intensity...")
    src_img = glob.glob('data/**/img/intensity', recursive=True)[0]
    src_lbl = glob.glob('data/**/lbs', recursive=True)[0]

print(f"Images source: {src_img}")
print(f"Labels source: {src_lbl}")

# Get sorted file lists
img_files = sorted(glob.glob(os.path.join(src_img, '*')))
lbl_files = sorted(glob.glob(os.path.join(src_lbl, '*.bmp')))

# Split: First 500 Test, Rest Train
test_pairs = list(zip(img_files[:500], lbl_files[:500]))
train_pairs = list(zip(img_files[500:], lbl_files[500:]))

print(f"Test Set: {len(test_pairs)} (First 500)")
print(f"Train Set: {len(train_pairs)} (Rest)")

# Copy to formatted directories
train_dir = os.path.abspath('data/Train')
test_dir = os.path.abspath('data/Test')

for pairs, dest in [(train_pairs, train_dir), (test_pairs, test_dir)]:
    os.makedirs(os.path.join(dest, '5d'), exist_ok=True)
    os.makedirs(os.path.join(dest, 'mask'), exist_ok=True)
    for img, mask in pairs:
        shutil.copy(img, os.path.join(dest, '5d'))
        shutil.copy(mask, os.path.join(dest, 'mask'))
print("Data preparation complete.")

/content/CrackSegDiff/data
Downloading...
From (original): https://drive.google.com/uc?id=1qnLMCeon7LJjT9H0ENiNF5sFs-F7-NvK
From (redirected): https://drive.google.com/uc?id=1qnLMCeon7LJjT9H0ENiNF5sFs-F7-NvK&confirm=t&uuid=9a5462ee-a712-44f5-a4e2-b58604b5d8d8
To: /content/CrackSegDiff/data/data.zip
100% 1.15G/1.15G [00:09<00:00, 118MB/s]
/content/CrackSegDiff
Organizing Data...
Images source: data/data/img/fused
Labels source: data/data/lbs
Test Set: 500 (First 500)
Train Set: 2000 (Rest)
Data preparation complete.


In [4]:
# 3. Train Model
import os
data_dir = os.path.abspath('data/Train')
out_dir = os.path.abspath('results/train_output')
os.makedirs(out_dir, exist_ok=True)

!python CrackSegDiff/segmentation_train.py --data_dir {data_dir} --out_dir {out_dir} --image_size 256 --num_channels 128 --class_cond False --num_res_blocks 2 --num_heads 1 --learn_sigma True --use_scale_shift_norm False --attention_resolutions 16 --diffusion_steps 1000 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --lr 5e-5 --batch_size 8 --save_interval 5000 --lr_anneal_steps 20000

/content/CrackSegDiff/CrackSegDiff/guided_diffusion/dpm_solver.py:41: SyntaxWarning: invalid escape sequence '\h'
  The `alphas_cumprod` is the \hat{alpha_n} arrays in the notations of DDPM. Specifically, DDPMs assume that
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Logging to /content/CrackSegDiff/results/train_output
creating data loader...
Your current directory :  /content/CrackSegDiff/data/Train
loading data from the directory : /content/CrackSegDiff/data/Train
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessi

In [5]:
# 4. Inference
import glob, os
# Use latest trained model
models = sorted(glob.glob('results/train_output/*.pt'))
model_path = models[-1] if models else "pretrained_weights/savedmodel100000.pt"
test_dir = os.path.abspath('data/Test')
print(f"Using model: {model_path}")

for modality in ['intensity', 'range', 'fused', 'filtered']:
    out_path = f"results/test_output_{modality}"
    os.makedirs(out_path, exist_ok=True)
    print(f"Testing {modality}...")
    !python CrackSegDiff/segmentation_sample.py --data_dir {test_dir} --out_dir {out_path} --model_path {model_path} --modality {modality} --image_size 256 --num_channels 128 --class_cond False --num_res_blocks 2 --num_heads 1 --learn_sigma True --use_scale_shift_norm False --attention_resolutions 16 --diffusion_steps 500 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --num_ensemble 1
    print(f"Done {modality}")

Using model: pretrained_weights/savedmodel100000.pt
Testing intensity...
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Logging to results/test_output_intensity
Your current directory :  /content/CrackSegDiff/data/Test
loading data from the directory : /content/CrackSegDiff/data/Test
creating model and diffusion...
[rank0]: Traceback (most recent call last):
[rank0]:   File "/content/CrackSegDiff/CrackSegDiff/segmentation_sample.py", line 245, in <module>
[rank0]:     main()
[rank0]:   File "/content/CrackSegDiff/CrackSegDiff/segmentation_sample.py", line 58, in main
[rank0]:     state_dict = dist_util.load_state_dict(args.model_path, map_location="cpu")
[rank0]:                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
[rank0]:   File "/co

In [6]:
# 5. Zip Results
!zip -r results.zip results

  adding: results/ (stored 0%)
  adding: results/test_output_fused/ (stored 0%)
  adding: results/test_output_fused/log.txt (deflated 6%)
  adding: results/test_output_fused/progress.csv (stored 0%)
  adding: results/test_output_range/ (stored 0%)
  adding: results/test_output_range/log.txt (deflated 4%)
  adding: results/test_output_range/progress.csv (stored 0%)
  adding: results/test_output_filtered/ (stored 0%)
  adding: results/test_output_filtered/log.txt (deflated 7%)
  adding: results/test_output_filtered/progress.csv (stored 0%)
  adding: results/test_output_intensity/ (stored 0%)
  adding: results/test_output_intensity/log.txt (deflated 5%)
  adding: results/test_output_intensity/progress.csv (stored 0%)
  adding: results/train_output/ (stored 0%)
  adding: results/train_output/log.txt (deflated 20%)
  adding: results/train_output/progress.csv (stored 0%)


In [7]:
import os

drive_path = '/content/drive/MyDrive/Datasets/FIND/Results/'
os.makedirs(drive_path, exist_ok=True)
!cp results.zip {drive_path}